# Sample Project Notebook

Goal of this project: simulate a 1m radius ball of Uranium in a box of air. represent the following tallies: 
- nu-fission: how many fission reactions occur
- absorption: how many absorption reactions occur
- sc

First, create the materials.

In [27]:
import openmc
import matplotlib.pyplot as plt
import numpy as np
"Notes:"
"-  units are in seconds, centimeters, and electronvolts."

'-  units are in seconds, centimeters, and electronvolts.'

In [28]:
uranium = openmc.Material(name="uranium")
uranium.add_nuclide("U238", .19, 'wo')
uranium.add_nuclide("U235", .8, 'wo')
uranium.add_nuclide("U234", .01, 'wo')
uranium.set_density("g/cm3", 19)

air = openmc.Material(name="air")
air.add_element("N", 0.78)
air.add_element("O", 0.22)
air.set_density("g/cm3", 0.001205)

materials = openmc.Materials([uranium, air])

export materials to xml file: 

In [29]:
materials.export_to_xml()

Second, create geometries: 

In [30]:
sphere = openmc.Sphere(r=1)
inside_sphere = -sphere
outside_sphere = +sphere

zregion = -openmc.ZPlane(5000, boundary_type="vacuum") & +openmc.ZPlane(-5000, boundary_type="vacuum")
yregion = -openmc.YPlane(5000, boundary_type="vacuum") & +openmc.YPlane(-5000, boundary_type="vacuum")
xregion = -openmc.XPlane(5000, boundary_type="vacuum") & +openmc.XPlane(-5000, boundary_type="vacuum")

inside_box = outside_sphere & zregion & yregion & xregion

# step 3: combine materials with geometry
uranium_ball = openmc.Cell()
uranium_ball.fill = uranium
uranium_ball.region = inside_sphere

box = openmc.Cell()
box.region = inside_box
box.fill = air

geometry = openmc.Geometry([uranium_ball, box])

Export geometries to xml file: 

In [31]:
geometry.export_to_xml()

Third, assign settings. These determine how many neutrons are coming from the source, and where the source is located in the model. 

In [32]:
settings = openmc.Settings()
settings.run_mode = "fixed source"
settings.particles = 10000
settings.batches = 100

source = openmc.IndependentSource()
source.space = openmc.stats.Point((0,0,0))
source.strength = 0.5
settings.source = source

Optionally, add surface filters that determine current. 

In [33]:
surfacefilter = openmc.SurfaceFilter(sphere)
cellfilter = openmc.CellFilter([box, uranium_ball])

Fourth, create tallies. these are counts of reactions that happen in a certain material. 

In [34]:
current_tally = openmc.Tally(name="current_tally")
current_tally.filters = [surfacefilter]
current_tally.scores = ['current']

flux_absorption_tally = openmc.Tally(name="flux_absorption_tally")
flux_absorption_tally.filters = [cellfilter]
flux_absorption_tally.scores = ['flux', 'absorption', 'nu-fission']

energy_bins = np.logspace(-5, 7.3, 50)
energyfilter = openmc.EnergyFilter(energy_bins)

spectrum_tally = openmc.Tally(name="spectrum_tally")
spectrum_tally.filters = [cellfilter, energyfilter]
spectrum_tally.scores = ['flux']

tallies_object = openmc.Tallies([flux_absorption_tally, current_tally, spectrum_tally])

Export settings and tallies to xml. 

In [35]:
tallies_object.export_to_xml()
settings.export_to_xml()

### return 1: xml sheets of everything. 

In [41]:
openmc.run()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

RuntimeError: Failed to open HDF5 file with mode 'w': summary.h5 Abort(-1) on node 0 (rank 0 in comm 0): application called MPI_Abort(MPI_COMM_WORLD, -1) - process 0

### return 2: create an image of the model 

In [ ]:
plot = openmc.Plot()
plot.basis = 'xz'  # or 'xy', 'yz'
plot.origin = (0, 0, 0)
plot.width = (20, 20)  # cm — zoom in near your sphere, not the full 10m box
plot.pixels = (800, 800)
plot.color_by = 'material'

plots = openmc.Plots([plot])
plots.export_to_xml()
openmc.plot_geometry()

### return 3: Energy-resolved flux spectrum
this plots how much flux based on the energy level of the neutron. we use the energy tally from step 4.

In [40]:
sp = openmc.StatePoint('statepoint.100.h5')

spectrum_tally = sp.get_tally(name="spectrum_tally")
df = spectrum_tally.get_pandas_dataframe()

# energy bin edges/midpoints for x-axis
e_low = df['energy low [eV]']
e_high = df['energy high [eV]']
e_mid = np.sqrt(e_low * e_high)          # geometric mean, standard for log bins
lethargy_width = np.log(e_high / e_low)  # bin width in lethargy units

fig, ax = plt.subplots(figsize=(8, 6))

for cell_id, cell_name in [(uranium_ball.id, 'Uranium sphere'), (box.id, 'Air')]:
    cell_df = df[df['cell'] == cell_id]
    # normalize flux by lethargy width so bin size doesn't distort shape
    flux_per_lethargy = cell_df['mean'] / lethargy_width[cell_df.index]
    ax.plot(e_mid[cell_df.index], flux_per_lethargy, label=cell_name, drawstyle='steps-mid')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Energy (eV)')
ax.set_ylabel('Flux per unit lethargy (n/cm²-s)')
ax.set_title('Neutron flux spectrum')
ax.legend()
plt.tight_layout()
plt.show()

/openmc_venv/lib/python3.12/site-packages/openmc/material.py:572: UserWarning: '' does not appear to be a nuclide name in GNDS format
  warnings.warn(str(e))


KeyError: ''